# SQL 2 (The Sequel)

**Seong Cho** · ACME Data Science Essentials, Lab 2

**Lab objective:** retrieve information spread across several tables of a SQL database with joins, grouping, ordering, wildcards, case expressions, and subqueries.

<sub>Lab specification from the 2017 *Volume 2* version of this lab, which later became Lab 2 of the *Data Science Essentials* lab manual of [Foundations of Applied Mathematics](https://foundations-of-applied-mathematics.github.io/) (BYU ACME), © Dr. J. Humpherys, used under [CC BY 3.0 US](http://creativecommons.org/licenses/by/3.0/us/). Problem statements are quoted from that 2017 version as worded in my lab template; the notes are condensed from the spec. Solutions are my own.</sub>

In [1]:
# solutions.py
# """Volume 3: SQL 2.
# <Name>
# <Class>
# <Date>
# """
import numpy as np
import matplotlib.pyplot as plt
import sqlite3 as sql
import csv

The examples and problems use `students.db`, the student database built in SQL 1: tables `MajorInfo (MajorID, MajorName)`, `CourseInfo (CourseID, CourseName)`, `StudentInfo (StudentID, StudentName, MajorID)` and `StudentGrades (StudentID, CourseID, Grade)`, where three students have a `NULL` major.

## Joining tables

A **join** combines rows from different tables on common attributes into a new, temporary table:

```sql
SELECT <alias.column, ...>
FROM <table> AS <alias> JOIN <table> AS <alias>, ...
ON <alias.column> == <alias.column>, ...
WHERE <condition>
```

The `ON` clause says how to match rows; joining $N$ tables typically takes $N-1$ conditions. Table aliases (`AS`) are essential once several tables are involved.

An **inner join** keeps only rows with an exact match in every table (an intersection). Joining `StudentInfo` and `MajorInfo` on `MajorID` therefore drops the three students whose `MajorID` is `NULL`, since `NULL` matches nothing.

In [2]:
with sql.connect("students.db") as conn:
    cur = conn.cursor()
    print(cur.execute("SELECT SI.StudentName, SI.StudentID "
                      "FROM StudentInfo AS SI INNER JOIN MajorInfo AS MI "
                      "ON SI.MajorID = MI.MajorID "
                      "WHERE MI.MajorName == 'Math';").fetchall())
conn.close()

[('Cassandra Holland', 341324754), ('Michelle Fernandez', 401767594)]


## Problem 1: B students

> **Problem 1.** Query the database for the list of the names of students who have a 'B' grade in any course. Return the list.

In [3]:
# Problem 1
def prob1(db_file="students.db"):
    """Query the database for the list of the names of students who have a
    'B' grade in any course. Return the list.

    Parameters:
        db_file (str): the name of the database to connect to.

    Returns:
        (list): a list of strings, each of which is a student name.
    """
    try:
        with sql.connect(db_file) as conn:
            cur = conn.cursor()
            cur.execute("SELECT SI.StudentName "
                        "FROM StudentInfo AS SI INNER JOIN StudentGrades AS SG "
                        "ON SI.StudentID = SG.StudentID "
                        "WHERE SG.Grade == 'B';")
            return [name[0] for name in cur.fetchall()]
    finally:
        conn.close()

In [4]:
prob1()

['Gilbert Chapman', 'Mercedes Hall', 'Alfonso Phelps']

## Outer joins and multiple joins

A **left outer join** keeps every row of the left-most table, plus the matching rows of the others; left rows without a match are padded with `NULL`. (SQLite has no `RIGHT OUTER JOIN`: `T1 RIGHT OUTER JOIN T2` is just `T2 LEFT OUTER JOIN T1`.)

Several joins of the same kind can be listed together (`INNER JOIN CourseInfo AS CI, StudentGrades SG ON ... AND ...`). Different kinds are chained one after another; the first join builds a temporary table and the next operates on it. Order matters: doing the left outer join with `MajorInfo` *before* an inner join with `StudentGrades` keeps students without a major in the result.

## Problem 2: Calculus grades

> **Problem 2.** Query the database for all tuples of the form `(Name, MajorName, Grade)` where `Name` is a student's name and `Grade` is their grade in Calculus. Only include results for students that are actually taking Calculus, but be careful not to exclude students who haven't declared a major.

In [5]:
# Problem 2
def prob2(db_file="students.db"):
    """Query the database for all tuples of the form (Name, MajorName, Grade)
    where 'Name' is a student's name and 'Grade' is their grade in Calculus.
    Only include results for students that are actually taking Calculus, but
    be careful not to exclude students who haven't declared a major.

    Parameters:
        db_file (str): the name of the database to connect to.

    Returns:
        (list): the complete result set for the query.
    """
    try:
        with sql.connect(db_file) as conn:
            cur = conn.cursor()
            cur.execute("SELECT SI.StudentName, MI.MajorName, SG.Grade "
                        "FROM StudentInfo AS SI LEFT OUTER JOIN MajorInfo AS MI "
                        "ON SI.MajorID = MI.MajorID "
                        "INNER JOIN StudentGrades AS SG "
                        "ON SI.StudentID = SG.StudentID "
                        "WHERE CourseID == 1;")
            return cur.fetchall()
    finally:
        conn.close()

In [6]:
prob2()

[('Gilbert Chapman', None, 'B'),
 ('Rene Cross', 'Writing', 'A'),
 ('Mercedes Hall', None, 'B'),
 ('Kristopher Tran', 'Science', 'C+'),
 ('Cassandra Holland', 'Math', 'A-'),
 ('Sammy Burke', 'Science', 'A')]

## Grouping data

`GROUP BY <attribute>` gathers rows into groups, which are then collapsed by an aggregate function: `AVG()`, `MIN()`, `MAX()`, `SUM()`, or `COUNT()` (`COUNT(*)` counts all rows of the group). Grouping works on the temporary table produced by a join as well.

`HAVING` filters *groups* the way `WHERE` filters rows. It often helps to alias the aggregated column with `AS` and refer to the alias later, e.g. `SELECT SI.StudentName, COUNT(*) AS num_courses ... GROUP BY SG.StudentID HAVING num_courses == 3`.

## Problem 3: Popular courses

> **Problem 3.** Query the database for the list of the names of courses that have at least 5 students enrolled in them.

In [7]:
# Problem 3
def prob3(db_file="students.db"):
    """Query the database for the list of the names of courses that have at
    least 5 students enrolled in them.

    Parameters:
        db_file (str): the name of the database to connect to.

    Returns:
        ((list): a list of strings, each of which is a course name.
    """
    try:
        with sql.connect(db_file) as conn:
            cur = conn.cursor()
            cur.execute("SELECT CI.CourseName "
                        "FROM CourseInfo AS CI INNER JOIN StudentGrades AS SG "
                        "ON CI.CourseID = SG.CourseID "
                        "GROUP BY SG.CourseID "
                        "HAVING COUNT(*) >= 5")
            return [name[0] for name in cur.fetchall()]
    finally:
        conn.close()

In [8]:
prob3()

['Calculus', 'English']

## Ordering result sets

`ORDER BY <attr> ASC|DESC, <attr> ASC|DESC, ...` sorts the result set by one or more attributes; it is always the very last clause of a query.

## Problem 4: Students per major

> **Problem 4.** Query the given database for tuples of the form `(MajorName, N)` where `N` is the number of students in the specified major. Sort the results in descending order by the counts `N`, then in alphabetic order by `MajorName`.

In [9]:
# Problem 4
def prob4(db_file="students.db"):
    """Query the given database for tuples of the form (MajorName, N) where N
    is the number of students in the specified major. Sort the results in
    descending order by the counts N, then in alphabetic order by MajorName.

    Parameters:
        db_file (str): the name of the database to connect to.

    Returns:
        (list): the complete result set for the query.
    """
    try:
        with sql.connect(db_file) as conn:
            cur = conn.cursor()
            cur.execute("SELECT MI.MajorName, COUNT(*) AS num_majors "
                        "FROM MajorInfo AS MI INNER JOIN StudentInfo as SI "
                        "ON SI.MajorID = MI.MajorID "
                        "GROUP BY SI.MajorID "
                        "ORDER BY num_majors DESC, MI.MajorName ASC")
            return cur.fetchall()
    finally:
        conn.close()

In [10]:
prob4()

[('Science', 3), ('Math', 2), ('Art', 1), ('Writing', 1)]

## Searching text with wildcards

`LIKE` in a `WHERE` clause matches patterns in a `TEXT` column, with two wildcards: `%` matches any number of characters and `_` matches exactly one. For example, `%Z_` matches strings ending in a `Z` and one more character, and `%i%` matches any string containing an `i`.

## Problem 5: Last names starting with C

> **Problem 5.** Query the database for tuples of the form `(StudentName, MajorName)` where the last name of the specified student begins with the letter C.

In [11]:
# Problem 5
def prob5(db_file="students.db"):
    """Query the database for tuples of the form (StudentName, MajorName) where
    the last name of the specified student begins with the letter C.

    Parameters:
        db_file (str): the name of the database to connect to.

    Returns:
        (list): the complete result set for the query.
    """
    try:
        with sql.connect(db_file) as conn:
            cur = conn.cursor()
            cur.execute("SELECT SI.StudentName, MI.MajorName "
                        "FROM StudentInfo AS SI LEFT OUTER JOIN MajorInfo as MI "
                        "ON SI.MajorID = MI.MajorID "
                        "WHERE StudentName LIKE '% C%';")
            return cur.fetchall()
    finally:
        conn.close()

In [12]:
prob5()

[('Gilbert Chapman', None),
 ('Roberta Cook', 'Science'),
 ('Rene Cross', 'Writing')]

## Case expressions and subqueries

A **case expression** maps column values. The *simple* form matches values (`CASE MajorID WHEN 1 THEN 'Mathematics' ... ELSE 'Undeclared' END`); the *searched* form tests a boolean expression at each step (`CASE WHEN MajorID IS NULL THEN 'Undeclared' ELSE 'Declared' END`).

A result set is just another table, so a whole query can be wrapped in parentheses and queried again (a **subquery**). Label the subquery's columns carefully with `AS` so the outer query can refer to them.

In [13]:
with sql.connect("students.db") as conn:
    cur = conn.cursor()
    print(cur.execute("SELECT majorstatus, COUNT(*) AS majorcount "
                      "FROM ( "
                          "SELECT StudentName, CASE "
                          "WHEN MajorID IS NULL THEN 'Undeclared' "
                          "ELSE 'Declared' END AS majorstatus "
                          "FROM StudentInfo) "
                      "GROUP BY majorstatus "
                      "ORDER BY majorcount DESC;").fetchall())
conn.close()

[('Declared', 7), ('Undeclared', 3)]


## Problem 6: GPA

> **Problem 6.** Query the database for tuples of the form `(StudentName, N, GPA)` where `N` is the number of courses that the specified student is in and `GPA` is the grade point average of the specified student according to the following point system.
>
> | | | | |
> |---|---|---|---|
> | A+, A = 4.0 | B = 3.0 | C = 2.0 | D = 1.0 |
> | A– = 3.7 | B– = 2.7 | C– = 1.7 | D– = 0.7 |
> | | B+ = 3.4 | C+ = 2.4 | D+ = 1.4 |
>
> Order the results from greatest GPA to least.

In [14]:
# Problem 6
def prob6(db_file="students.db"):
    """Query the database for tuples of the form (StudentName, N, GPA) where N
    is the number of courses that the specified student is in and 'GPA' is the
    grade point average of the specified student according to the following
    point system.

        A+, A  = 4.0    B  = 3.0    C  = 2.0    D  = 1.0
            A- = 3.7    B- = 2.7    C- = 1.7    D- = 0.7
            B+ = 3.4    C+ = 2.4    D+ = 1.4

    Order the results from greatest GPA to least.

    Parameters:
        db_file (str): the name of the database to connect to.

    Returns:
        (list): the complete result set for the query.
    """
    try:
        with sql.connect(db_file) as conn:
            cur = conn.cursor()
            cur.execute("SELECT StudentName, COUNT(*), AVG(GradePoint) AS GPA "
                        "FROM ( "
                            "SELECT SI.StudentName, CASE SG.Grade "
                                "WHEN 'A+' THEN 4.0 "
                                "WHEN 'A' THEN 4.0 "
                                "WHEN 'A-' THEN 3.7 "
                                "WHEN 'B+' THEN 3.4 "
                                "WHEN 'B' THEN 3.0 "
                                "WHEN 'B-' THEN 2.7 "
                                "WHEN 'C+' THEN 2.4 "
                                "WHEN 'C' THEN 2.0 "
                                "WHEN 'C-' THEN 1.7 "
                                "WHEN 'D+' THEN 1.4 "
                                "WHEN 'D' THEN 1.0 "
                                "WHEN 'D-' THEN 0.7 "
                                "ELSE 0 END AS GradePoint "
                            "FROM StudentInfo AS SI INNER JOIN StudentGrades AS SG "
                            "ON SI.StudentID = SG.StudentID) "
                        "GROUP BY StudentName "
                        "ORDER BY GPA DESC;")
            return cur.fetchall()
    finally:
        conn.close()

In [15]:
prob6()

[('Rene Cross', 1, 4.0),
 ('Sammy Burke', 2, 3.85),
 ('Gilbert Chapman', 3, 3.6666666666666665),
 ('Kristopher Tran', 2, 3.2),
 ('Mercedes Hall', 3, 2.6666666666666665),
 ('Cameron Kim', 3, 2.566666666666667),
 ('Alfonso Phelps', 2, 2.5),
 ('Michelle Fernandez', 2, 2.35),
 ('Cassandra Holland', 2, 2.2),
 ('Roberta Cook', 1, 2.0)]